# The CMB as a perfect blackbody (COBE FIRAS)

*cosmology · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/cmb_blackbody)

COBE's FIRAS instrument measured the cosmic microwave background at 43 frequencies and found the most perfect blackbody ever seen. A batch of four models (Planck law alone, with a Bose–Einstein chemical potential $\mu$, with a Compton $y$-distortion, with both) is fitted to the real FIRAS table in one tensor, with the Jacobian from autograd. The fit gives $T \approx 2.725$ K with $\chi^2 \approx 45$ for 42 degrees of freedom, residuals below $10^{-4}$ of the peak, distortions consistent with zero ($|\mu| \lesssim 10^{-4}$, $|y| \lesssim 10^{-5}$), and from $T$ about 411 photons per cm³.

**Vector form**

$$I_\nu = \frac{2h\nu^3}{c^2}\left[\frac{1}{e^{x+\mu}-1} + y\,\frac{x e^x}{(e^x-1)^2}\Big(x\coth\frac{x}{2} - 4\Big)\right],\qquad x = \frac{h\nu}{kT},\qquad \chi^2 = \sum_f \Big(\frac{d_f - I_f}{\sigma_f}\Big)^2,\qquad n_\gamma = \frac{2\zeta(3)}{\pi^2}\Big(\frac{kT}{\hbar c}\Big)^3$$

**Index (tensor) form**

$$I_{bf} = 2hc\,\tilde\nu_f^3\big[(e^{x_{bf}+\mu_b}-1)^{-1} + y_b\,Y(x_{bf})\big],\qquad J_{bfk} = \frac{\partial I_{bf}}{\partial\theta_{bk}},\qquad (J_{bfi}J_{bfj}/\sigma_f^2)\,\delta\theta_{bj} = J_{bfi}\,r_{bf}/\sigma_f^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\tilde\nu_f,\ d_f,\ \sigma_f$ | `load_firas(device)` | (F,) | the 43 FIRAS wavenumbers (cm$^{-1}$), intensities and $1\sigma$ errors (MJy/sr) |
| $\theta_{bk}$ | `initial_params(T_init, device)` | (B, 3) | one row $(T, \mu, y)$ per model $b$ of the batch |
| $x_{bf}$ | `C2 * nu / T` | (B, F) | $h\nu/kT$ by broadcasting $(B,1)$ against $(F,)$ |
| $I_{bf}$ | `spectrum(nu, theta)` | (B, F) | Bose–Einstein occupation plus the Kompaneets $y$ term, times $2hc\tilde\nu^3$ |
| $J_{bfk}$ | `jacobian(nu, theta)` | (B, F, 3) | autograd of $\sum_b I_{bf}$: the rows are independent, so one call gives every model's Jacobian |
| $J_{bfi}J_{bfj}/\sigma_f^2$ | `einsum('bfi,bfj->bij', A, A)` | (B, 3, 3) | repeated $f$ ⇒ summed: the Fisher matrix; its inverse is the covariance |
| $\delta\theta_{bj}$ | `linalg.solve(damped, grad)` | (B, 3) | one Levenberg–Marquardt step for every model at once |
| $\chi^2_b$ | `einsum('bf,bf->b', r, r)` | (B,) | goodness of fit per model |

In 1990 FIRAS (the Far-Infrared Absolute Spectrophotometer on NASA's COBE) compared the sky with an external blackbody calibrator and found no difference; the final spectrum (Fixsen et al. 1996) deviates from a Planck curve by less than 50 parts per million of the peak. John Mather and George Smoot shared the 2006 Nobel Prize in Physics "for their discovery of the blackbody form and anisotropy of the cosmic microwave background radiation". A blackbody this perfect means the early universe was hot, dense and in thermal equilibrium: energy released after redshift $z \sim 2\times10^6$ would have left a chemical potential $\mu$ (Bose–Einstein spectrum, $z \gtrsim 2\times10^5$) or a Compton $y$-distortion (hot electrons boosting photons, $z \lesssim 5\times10^4$; the same shape as the Sunyaev–Zel'dovich effect, with its null at $x = 3.83$ or 217 GHz, and it conserves photon number). Our fit, with only the diagonal errors, gives 95 % limits of about $|\mu| < 7\times10^{-5}$ and $|y| < 6\times10^{-6}$; the published FIRAS limits, $|\mu| < 9\times10^{-5}$ and $|y| < 1.5\times10^{-5}$, are wider because they use the full correlated covariance and include systematic (calibration and Galactic) uncertainties. Read the temperature with care: the fit pins $T$ to about $10^{-5}$ K statistically, but the absolute scale comes from the thermometry of the calibrator, and the LAMBDA table writes the spectrum as a 2.725 K blackbody plus the published residuals, so $T = 2.725$ K is built in; the best absolute value today is $2.72548 \pm 0.00057$ K (Fixsen 2009). From $T$ follow $n_\gamma \approx 411$ cm$^{-3}$ and $u_\gamma = aT^4 \approx 0.26$ eV cm$^{-3}$. In the animation the left panel shows the fit converging from the starting guess (error bars magnified 400 times, as in the famous plot, otherwise invisible); the right panel then zooms into the residuals in units of $10^{-4}$ of the peak, with the shapes a $\mu$ or $y$ at our 95 % limit would leave after re-fitting $T$. Data: Table 4 of D. J. Fixsen et al., ApJ 473, 576 (1996), from NASA's LAMBDA archive (https://lambda.gsfc.nasa.gov/data/cobe/firas/monopole_spec/firas_monopole_spec_v1.txt), public domain, bundled as data/firas_monopole.csv.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/cmb_blackbody/main.py
"""Main loop: FIRAS data → batched fit of (T, μ, y) → limits and derived densities → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

PUBLISHED = {"mu": 9e-5, "y": 1.5e-5}  # Fixsen et al. 1996, 95 % confidence


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    data = tensors.load_firas(dev)
    nu, intensity, sigma = data["nu"], data["intensity"], data["sigma"]
    fit = solve.fit(nu, intensity, sigma, T_init=params["T_init"], steps=params["steps"])

    print(f"FIRAS monopole: {len(nu)} frequencies from {nu[0]:.2f} to {nu[-1]:.2f} cm⁻¹ (Fixsen et al. 1996)")
    for b, name in enumerate(tensors.MODELS):
        T, mu, y = fit["params"][b].tolist()
        sT, smu, sy = fit["sigma"][b].tolist()
        line = f"{name:10s} T = {T:.6f} ± {sT:.6f} K"
        line += f"   μ = {mu:+.2e} ± {smu:.1e}" if smu > 0 else ""
        line += f"   y = {y:+.2e} ± {sy:.1e}" if sy > 0 else ""
        print(f"{line}   χ² = {fit['chi2'][b]:.1f} for {int(fit['dof'][b])} dof")
    mu_lim, y_lim = solve.upper_limits(fit)
    print(
        f"95 % limits: |μ| < {mu_lim:.1e} (published {PUBLISHED['mu']:.0e}),"
        f"  |y| < {y_lim:.1e} (published {PUBLISHED['y']:.1e})"
    )
    print("(ours use only the diagonal errors; the published limits include correlated errors and systematics)")
    T = fit["params"][0, 0]
    n_gamma, u_gamma = equation.photon_number_density(T).item(), equation.energy_density(T).item()
    print(f"from T: photon number density n_γ = {n_gamma:.1f} cm⁻³, energy density u_γ = {u_gamma:.4f} eV cm⁻³")

    # what the plot needs: the T-only model at each fit step until it stops moving
    chi2_t = fit["chi2_history"][:, 0]
    moving = (chi2_t[:-1] - chi2_t[1:]).abs() > 1e-9 * chi2_t[1:]
    n_show = int(moving.nonzero().max()) + 2 if moving.any() else 1
    hist = fit["history"][:n_show, 0]  # (K, 3)
    nu_fine = torch.linspace(0.5, 23.0, 300, dtype=nu.dtype, device=dev)
    T_best, mu_best, y_best = fit["params"][0, 0].item(), 0.0, 0.0
    view = {
        "nu": nu, "data": intensity, "sigma": sigma, "nu_fine": nu_fine,
        "curves": equation.spectrum(nu_fine, hist), "model_at_data": equation.spectrum(nu, hist),
        "T": hist[:, 0], "chi2": chi2_t[:n_show], "dof": fit["dof"][0].item(),
        "mu_limit": solve.distortion_after_refit(nu, sigma, T_best, mu_best - mu_lim, y_best),
        "y_limit": solve.distortion_after_refit(nu, sigma, T_best, mu_best, y_best + y_lim),
    }  # fmt: skip
    view = {k: v.cpu().numpy() if torch.is_tensor(v) else v for k, v in view.items()}
    view["mu_lim"], view["y_lim"] = -mu_lim, y_lim
    view["summary"] = (
        f"T = {T_best:.5f} K (stat. ±{fit['sigma'][0, 0]:.0e})\n"
        f"χ²/dof = {fit['chi2'][0]:.1f}/{int(fit['dof'][0])}\n"
        f"|μ| < {mu_lim:.0e}  (Fixsen: 9e-05)\n|y| < {y_lim:.0e}  (Fixsen: 1.5e-05)\n"
        f"n_γ = {n_gamma:.0f} cm⁻³, u_γ = {u_gamma:.3f} eV/cm³"
    )
    return plot.animate(view, out_dir / "cmb_blackbody.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/cmb_blackbody/equation.py
r"""Physics → tensor translation: the Planck law and its two small distortions.

FIRAS measured the sky intensity I_ν against wavenumber ν̃ = ν/c in cm⁻¹. With x = hν/kT = c₂ ν̃ / T
(c₂ = hc/k = 1.4388 cm K) a blackbody has the photon occupation n = 1/(eˣ − 1) and

    I_ν = 2hν³/c² · n = 2hc ν̃³ · n          (per Hz per steradian)

Two ways energy injected into the early universe would bend this curve:

    μ-distortion (z ≳ 2×10⁵, Bose–Einstein equilibrium):  n = 1 / (e^{x + μ} − 1)
    y-distortion (z ≲ 5×10⁴, Compton scattering off hot electrons, Kompaneets):
        Δn = y · x eˣ/(eˣ − 1)² · (x coth(x/2) − 4)

so the full model, for a batch of B parameter rows θ_b = (T_b, μ_b, y_b) and F frequencies, is

    I_{bf} = 2hc ν̃_f³ [ 1/(e^{x_{bf} + μ_b} − 1) + y_b Y(x_{bf}) ],     x_{bf} = c₂ ν̃_f / T_b

No index is summed: the batch is pure broadcasting (B, 1) against (F,).
"""

import math

import torch

H_PLANCK = 6.62607015e-34  # J s
K_B = 1.380649e-23  # J/K
C_LIGHT = 299792458.0  # m/s
C2 = H_PLANCK * C_LIGHT / K_B * 100  # cm K, second radiation constant: x = C2 ν̃ / T
PREF = 2 * H_PLANCK * C_LIGHT * 100**3 * 1e20  # 2hc ν̃³ in MJy/sr for ν̃ in cm⁻¹ (1 MJy = 1e-20 W m⁻² Hz⁻¹)
ZETA3 = 1.2020569031595942
EV = 1.602176634e-19  # J


def planck(nu: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    """Blackbody I_ν in MJy/sr at wavenumber nu (cm⁻¹) and temperature T (K); any broadcastable shapes."""
    x = C2 * nu / T
    return PREF * nu**3 / torch.expm1(x)


def y_shape(x: torch.Tensor) -> torch.Tensor:
    r"""Y(x) = Δn / y = x eˣ/(eˣ − 1)² · (x coth(x/2) − 4), the Compton (Sunyaev–Zel'dovich) distortion."""
    return x * torch.exp(x) / torch.expm1(x) ** 2 * (x / torch.tanh(x / 2) - 4)


def spectrum(nu: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """I_{bf} for parameter rows theta (B, 3) = (T, μ, y) and wavenumbers nu (F,) → (B, F) in MJy/sr."""
    T, mu, y = theta[:, 0:1], theta[:, 1:2], theta[:, 2:3]  # (B, 1) each, broadcast against (F,)
    x = C2 * nu / T  # (B, F)
    occupation = 1 / torch.expm1(x + mu) + y * y_shape(x)  # (B, F) Bose–Einstein + Kompaneets
    return PREF * nu**3 * occupation


def chi2(model: torch.Tensor, data: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    r"""χ²_b = Σ_f ((d_f − I_{bf}) / σ_f)²: repeated f summed.   (B, F) → (B,)"""
    r = (data - model) / sigma
    return torch.einsum("bf,bf->b", r, r)


def photon_number_density(T: torch.Tensor) -> torch.Tensor:
    r"""n_γ = (2ζ(3)/π²) (kT/ħc)³ in cm⁻³ (≈ 411 for the CMB)."""
    hbar = H_PLANCK / (2 * math.pi)
    return 2 * ZETA3 / math.pi**2 * (K_B * T / (hbar * C_LIGHT)) ** 3 * 1e-6


def energy_density(T: torch.Tensor) -> torch.Tensor:
    r"""u_γ = (π²/15) (kT)⁴/(ħc)³ = a T⁴ in eV cm⁻³ (≈ 0.26 for the CMB)."""
    hbar = H_PLANCK / (2 * math.pi)
    return math.pi**2 / 15 * (K_B * T) ** 4 / (hbar * C_LIGHT) ** 3 / EV * 1e-6


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/cmb_blackbody/tensors.py
"""Build the concrete tensors: the bundled FIRAS spectrum and the batch of models to fit.

data/firas_monopole.csv is Table 4 of Fixsen et al. (1996), as served by NASA LAMBDA (public domain).
"""

from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data" / "firas_monopole.csv"
MODELS = ["T only", "T + μ", "T + y", "T + μ + y"]
# which of (T, μ, y) each model may vary: (B, 3)
FREE = torch.tensor([[1, 0, 0], [1, 1, 0], [1, 0, 1], [1, 1, 1]], dtype=torch.float64)


def load_firas(device: torch.device) -> dict:
    """Wavenumber (cm⁻¹), intensity and 1σ uncertainty (MJy/sr), each (F,) with F = 43, in float64."""
    table = np.loadtxt(DATA, delimiter=",", comments="#", skiprows=4)  # (F, 5)
    col = {name: torch.tensor(table[:, k], dtype=torch.float64, device=device) for k, name in enumerate(
        ["nu", "intensity", "residual", "sigma", "galaxy"])}  # fmt: skip
    for name in ("residual", "sigma", "galaxy"):
        col[name] = col[name] * 1e-3  # kJy/sr → MJy/sr
    return col


def initial_params(T_init: float, device: torch.device) -> torch.Tensor:
    """One row (T, μ, y) per model, all starting from the same guess with no distortion: (B, 3)."""
    theta = torch.zeros(len(MODELS), 3, dtype=torch.float64, device=device)
    theta[:, 0] = T_init
    return theta


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/cmb_blackbody/solve.py
"""Fit every model of the batch at once: Levenberg–Marquardt steps with the Jacobian from autograd.

Each step linearises I_{bf}(θ_b) around the current θ_b. The Jacobian J_{bfk} = ∂I_{bf}/∂θ_{bk} comes from one
autograd call: the rows of the batch are independent, so differentiating Σ_b I_{bf} with respect to θ (B, 3)
gives every model's own Jacobian. A 3 × 3 solve per model (batched) then gives the step.
"""

import torch

from . import equation, tensors


def jacobian(nu: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """J_{bfk} = ∂I_{bf}/∂θ_{bk}: (B, F, 3) by autograd."""
    J = torch.autograd.functional.jacobian(lambda th: equation.spectrum(nu, th).sum(0), theta)  # (F, B, 3)
    return J.permute(1, 0, 2)


def fit(nu, data, sigma, T_init: float = 2.5, steps: int = 40, free: torch.Tensor | None = None) -> dict:
    """data (F,) or (B, F). Returns the fitted θ (B, 3), 1σ errors, χ², degrees of freedom and the history."""
    free = (tensors.FREE if free is None else free).to(nu.device)  # (B, 3) 1 = fitted, 0 = held at 0
    theta = tensors.initial_params(T_init, nu.device)
    lam = torch.full((len(theta),), 1e2, dtype=theta.dtype, device=nu.device)  # damping per model
    eye = torch.eye(3, dtype=theta.dtype, device=nu.device)
    mask = torch.einsum("bi,bj->bij", free, free)  # (B, 3, 3) couplings between fitted parameters only
    pinned = torch.diag_embed(1 - free)  # puts 1 on the diagonal of a held parameter so the solve stays regular

    def normal_equations(th):
        A = jacobian(nu, th) / sigma[:, None]  # (B, F, 3) whitened Jacobian
        r = (data - equation.spectrum(nu, th)) / sigma  # (B, F) whitened residual
        fisher = torch.einsum("bfi,bfj->bij", A, A) * mask + pinned  # repeated f summed: AᵀA
        grad = torch.einsum("bfi,bf->bi", A, r) * free  # Aᵀr
        return fisher, grad

    chi2 = equation.chi2(equation.spectrum(nu, theta), data, sigma)
    history, chi2_history = [theta.clone()], [chi2.clone()]
    for _ in range(steps):
        fisher, grad = normal_equations(theta)
        damped = fisher + lam[:, None, None] * fisher * eye  # Marquardt: scale-free damping of the diagonal
        step = torch.linalg.solve(damped, grad[..., None])[..., 0]  # (B, 3)
        trial = theta + step * free
        trial[:, 0] = trial[:, 0].clamp(min=0.5)  # a temperature step never goes negative
        chi2_trial = equation.chi2(equation.spectrum(nu, trial), data, sigma)
        better = chi2_trial < chi2  # (B,) accept per model
        theta = torch.where(better[:, None], trial, theta)
        chi2 = torch.where(better, chi2_trial, chi2)
        lam = torch.where(better, lam / 3, lam * 4).clamp(1e-9, 1e9)
        history.append(theta.clone())
        chi2_history.append(chi2.clone())

    fisher, _ = normal_equations(theta)
    cov = torch.linalg.inv(fisher) * mask  # (B, 3, 3) covariance of the fitted parameters
    return {
        "params": theta,
        "sigma": torch.diagonal(cov, dim1=-2, dim2=-1).sqrt(),  # (B, 3) 1σ, zero where held
        "cov": cov,
        "chi2": chi2,
        "dof": nu.shape[0] - free.sum(-1),  # (B,)
        "history": torch.stack(history),  # (steps + 1, B, 3)
        "chi2_history": torch.stack(chi2_history),  # (steps + 1, B)
    }


def upper_limits(fit: dict, z: float = 1.96) -> tuple[float, float]:
    """95 % limits |μ| < |μ̂| + 1.96 σ_μ (T + μ model) and |y| < |ŷ| + 1.96 σ_y (T + y model)."""
    p, s = fit["params"], fit["sigma"]
    return (p[1, 1].abs() + z * s[1, 1]).item(), (p[2, 2].abs() + z * s[2, 2]).item()


def distortion_after_refit(nu, sigma, T: float, mu: float, y: float) -> torch.Tensor:
    """What a (μ, y) distortion leaves in the data once T is re-fitted: ΔI minus its best (weighted) T-shift, (F,)."""
    base = torch.tensor([[T, 0.0, 0.0]], dtype=nu.dtype, device=nu.device)
    bent = torch.tensor([[T, mu, y]], dtype=nu.dtype, device=nu.device)
    delta = (equation.spectrum(nu, bent) - equation.spectrum(nu, base))[0]  # (F,)
    dB_dT = jacobian(nu, base)[0, :, 0]  # (F,) the shape of a pure temperature change
    w = sigma**-2
    shift = torch.einsum("f,f,f->", w, dB_dT, delta) / torch.einsum("f,f,f->", w, dB_dT, dB_dT)
    return delta - shift * dB_dT


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/cmb_blackbody/plot.py
"""Animate the fit converging on the FIRAS points, then zoom into the residuals in units of 10⁻⁴ of the peak."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

GHZ_PER_CM = 29.9792458  # 1 cm⁻¹ = 29.98 GHz


def animate(view: dict, path: Path, hold: int = 6, zoom: int = 30) -> Path:
    """view: numpy arrays from main.py. The first phase steps through the fit, the second shrinks the residual axis."""
    nu, data, err = view["nu"], view["data"], view["sigma"]
    curves, T_hist, chi2_hist = view["curves"], view["T"], view["chi2"]  # (K, G), (K,), (K,)
    nu_fine = view["nu_fine"]
    model_at_data = view["model_at_data"]  # (K, F) T-only model at the data points, per shown iteration
    peak = data.max()
    res = (data[None] - model_at_data) / peak * 1e4  # (K, F) residuals in units of 10⁻⁴ of the peak
    n_iter = len(T_hist)
    schedule = [k for k in range(n_iter) for _ in range(hold)] + [n_iter - 1] * zoom
    final_span = 3.5
    start_span = max(final_span, float(np.abs(res[-1]).max()) * 1.15)

    fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(8.6, 4), gridspec_kw={"width_ratios": [1.1, 1]})
    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1.15 * max(peak, curves.max()))
    ax.set_xlabel("wavenumber ν̃ (cm⁻¹)")
    ax.set_ylabel("intensity I_ν (MJy/sr)")
    ax.errorbar(
        nu,
        data,
        yerr=400 * err,
        fmt="o",
        ms=3,
        color="#111827",
        ecolor="#6b7280",
        lw=1,
        label="FIRAS (error bars ×400)",
    )
    line = ax.plot([], [], "-", color=ACCENT[0], lw=2, label="Planck law, fitted T")[0]
    ax.legend(fontsize=8, loc="upper right")
    top = ax.secondary_xaxis("top", functions=(lambda c: c * GHZ_PER_CM, lambda g: g / GHZ_PER_CM))
    top.set_xlabel("frequency (GHz)", fontsize=8)
    title = fig.suptitle("", fontsize=10)

    ax_r.set_xlim(0, 23)
    ax_r.axhline(0, color="#9ca3af", lw=1)
    ax_r.set_xlabel("wavenumber ν̃ (cm⁻¹)")
    ax_r.set_ylabel("(data − blackbody) / peak  (×10⁻⁴)")
    bars = ax_r.errorbar(nu, res[0], yerr=err / peak * 1e4, fmt="o", ms=3, color="#111827", ecolor="#6b7280", lw=1)
    mu_line = ax_r.plot(nu, view["mu_limit"] / peak * 1e4, "--", color=ACCENT[1], lw=1.5, alpha=0)[0]
    y_line = ax_r.plot(nu, view["y_limit"] / peak * 1e4, "--", color=ACCENT[3], lw=1.5, alpha=0)[0]
    info = ax_r.text(0.03, 0.97, "", transform=ax_r.transAxes, va="top", fontsize=7.5, family="monospace")
    ax_r.set_title("residuals", fontsize=9)

    def update(i: int):
        k = schedule[i]
        line.set_data(nu_fine, curves[k])
        title.set_text(f"fit step {k}:  T = {T_hist[k]:.5f} K   χ²/dof = {chi2_hist[k] / view['dof']:.3g}")
        bars.lines[0].set_ydata(res[k])
        segs = [np.array([[x, r - e], [x, r + e]]) for x, r, e in zip(nu, res[k], err / peak * 1e4)]
        bars.lines[2][0].set_segments(segs)
        z = 0.0 if i < n_iter * hold else (i - n_iter * hold + 1) / zoom  # 0 → 1 during the zoom phase
        if z == 0:  # while fitting: follow the residuals
            span = max(final_span, float(np.abs(res[k]).max()) * 1.15)
        else:  # then zoom in to a few 10⁻⁴ (the last, noisiest points may leave the frame)
            span = max(final_span, start_span ** (1 - z) * final_span**z)
        ax_r.set_ylim(-span, span)
        shown = z > 0.5
        mu_line.set_alpha(0.9 if shown else 0)
        y_line.set_alpha(0.9 if shown else 0)
        mu_line.set_label(f"μ = {view['mu_lim']:.1e} (our 95 % limit)" if shown else "_")
        y_line.set_label(f"y = {view['y_lim']:.1e} (our 95 % limit)" if shown else "_")
        if shown and ax_r.get_legend() is None:
            ax_r.legend(fontsize=7, loc="lower left")
        info.set_text(view["summary"] if shown else "")

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return save_gif(fig, update, frame_indices(len(schedule), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T_init': 2.5,
    'steps': 30
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.cmb_blackbody")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.cmb_blackbody.main")
problem_dir = Path("problems/cosmology/cmb_blackbody")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
